In [1]:
pip install geopandas rasterio rasterstats pyproj


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
#pip install geopandas rasterio rasterstats pandas numpy shapely
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping
import numpy as np
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [6]:
tracts_path = "data_raw/tl_2022_37_tract"
tract_id_col = "GEOID"                   # id column to keep
out_csv = "data_processed/selected_tracts_tcc_summary.csv"

# your lists (as provided)
ranked_most = [
    "37149920304",
    "37021003203",
    "37161960301"
]

study_tracts = [
    "37183054402",
    "37135011103",
    "37063001809",
    "37119003802",
    "37183054108",
    "37119006212"
]

rtp_focus = [
    "37063001900",
    "37183053109",
    "37063002100",
    "37183053430",
    "37183052810",
    "37135011210",
    "37135010902",
    "37063002020",
    "37183053428",
    "37183053419"
]

charlotte_focus = [
    "37119006212",
    "37119002702",
    "37119003902",
    "37119004000",
    "37119003600",
    "37119005856",
    "37119003110",
    "37119004800",
    "37119006222",
    "37119005100"
]

wnc_focus = [
    "37149920304",
    "37111970700",
    "37111970800",
    "37021003204",
    "37111970500",
    "37021003205",
    "37089930100",
    "37021003105",
    "37161960301",
    "37161960202"
]

In [7]:
# combine unique list of GEOIDs you want
selected_geoids = list({*ranked_most, *study_tracts, *rtp_focus, *charlotte_focus, *wnc_focus})

# remote TIFF inside ZIP (from your gdalinfo output)
tcc_url = (
    "/vsizip//vsicurl/https://data.fs.usda.gov/geodata/"
    "rastergateway/treecanopycover/docs/v2023-5/"
    "nlcd_tcc_CONUS_2023_v2023-5_wgs84.zip/"
    "nlcd_tcc_conus_wgs84_v2023-5_20230101_20231231.tif"
)

# ------------------ load tracts ------------------
tracts = gpd.read_file(tracts_path)

# ensure GEOID is string, zero-padded if needed (standard US GEOIDs are strings)
tracts[tract_id_col] = tracts[tract_id_col].astype(str)

# filter to only selected GEOIDs
selected_df = tracts[tracts[tract_id_col].isin(selected_geoids)].copy()

# report missing
found = set(selected_df[tract_id_col].tolist())
missing = sorted(set(selected_geoids) - found)
if missing:
    print("Warning: the following GEOIDs were NOT found in your tract file:", missing)

if selected_df.empty:
    raise SystemExit("No tracts found. Check tract_id_col and tracts_path.")

# ------------------ open raster ------------------
with rasterio.Env():
    with rasterio.open(tcc_url) as src:
        raster_crs = src.crs
        print("Raster CRS:", raster_crs)
        # reproject tracts if needed
        if selected_df.crs != raster_crs:
            selected_df = selected_df.to_crs(raster_crs)

        results = []
        for idx, row in selected_df.iterrows():
            geoid = row[tract_id_col]
            geom = row.geometry
            try:
                # mask: reads only the window overlapping the polygon (streamed)
                out_image, out_transform = mask(src, [geom], crop=True, all_touched=False)
                arr = out_image[0]  # single band
            except Exception as e:
                print(f"Error reading raster for GEOID {geoid}: {e}")
                results.append({
                    "GEOID": geoid,
                    "mean_canopy_pct": None,
                    "pct_pixels_ge_50": None,
                    "valid_pixel_count": 0
                })
                continue

            # NLCD TCC uses 255 as nodata
            nodata_val = 255
            valid_mask = (arr != nodata_val)
            valid_pixels = arr[valid_mask]

            if valid_pixels.size == 0:
                mean_canopy = None
                pct_ge_50 = None
                valid_count = 0
            else:
                # values are 0..100
                mean_canopy = float(valid_pixels.mean())
                pct_ge_50 = float((valid_pixels >= 50).sum() / valid_pixels.size * 100.0)
                valid_count = int(valid_pixels.size)

            results.append({
                "GEOID": geoid,
                "mean_canopy_pct": mean_canopy,
                "pct_pixels_ge_50": pct_ge_50,
                "valid_pixel_count": valid_count
            })

# ------------------ save & print ------------------
df_out = pd.DataFrame(results).sort_values("GEOID").reset_index(drop=True)
# add which group(s) each GEOID belongs to (optional)
def groups_for(g):
    groups = []
    if g in ranked_most: groups.append("ranked_most")
    if g in study_tracts: groups.append("study_tracts")
    if g in rtp_focus: groups.append("rtp_focus")
    if g in charlotte_focus: groups.append("charlotte_focus")
    if g in wnc_focus: groups.append("wnc_focus")
    return ";".join(groups)

df_out["groups"] = df_out["GEOID"].apply(groups_for)

df_out.to_csv(out_csv, index=False)
print(f"\nSaved {len(df_out)} tract summaries to: {out_csv}\n")
print(df_out)

# quick summary stats
print("\nSummary of mean_canopy_pct (non-null values):")
print(df_out["mean_canopy_pct"].dropna().describe())

Raster CRS: PROJCS["Albers_Conical_Equal_Area",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Saved 36 tract summaries to: data_processed/selected_tracts_tcc_summary.csv

          GEOID  mean_canopy_pct  pct_pixels_ge_50  valid_pixel_count  \
0   37021003105        77.917985         85.866176              26617   
1   37021003203        67.491775         74.534527              32279   
2   37021003204        71.482679         80.268735              95782   
3   37021003205     